# S6E8 Diversity Beats Strength（解説付き写し）

| 項目 | 内容 |
|---|---|
| コンペ | [Predicting Smartphone Addiction (Playground Series S6E8)](https://www.kaggle.com/competitions/playground-series-s6e8) |
| 元notebook | [S6E8 Diversity Beats Strength](https://www.kaggle.com/code/adarsh1077/s6e8-diversity-beats-strength) |
| 原著者 | Adarsh Aleti (@adarsh1077) |
| Public Score | **0.97113**（Version 8 / 41 votes、本日時点でこのコンペの最上位クラス） |
| ライセンス | Apache 2.0 |

> **断り書き**：これは学習目的の「解説付き写し」です。原著者のコードは変更していませんが、実行結果（出力）は含めていません。

## 手法の概要

スマートフォン依存の有無（2値）を予測するタスク。順位は **AUC** で決まります。

このnotebookが面白いのは、**新しいモデルを作った話ではなく「アンサンブルの中身を計測した話」**である点です。著者は公開されている6つのOOFライブラリ（他の参加者が公開した予測配列）から155本、自作22本、合計177本の予測を集め、それらを1つのロジスティック回帰で束ねています。そして「何がスコアを上げたのか」を**同じfoldで測って**表にしています。

結論は表題どおり、**「強さ（accuracy）ではなく、ばらけ方（diversity）が効く」**。メタモデルが最も大きな重みを与えたのは、単体AUCが低いモデル — 特に**target encodingを外したモデル**でした。他の全員がtarget encodingを使っている以上、それを外したモデルだけが「別の見方」を持ち込めるからです。

さらに著者は、**このコンペのAUCの理論的な上限**を約 **0.9701** と推定しています（後述）。これは「これ以上いくら頑張っても伸びない」というラインで、時間の使い方を決めるうえで非常に実用的な情報です。

## 評価指標

- **指標**：**ROC-AUC**。「無作為に選んだ陽性1件のスコアが、無作為に選んだ陰性1件のスコアより高い確率」。
- **なぜAUCか**：この手のターゲットは陽性率が偏りがちで、正解率（accuracy）だと「全部陰性と答える」だけで高い値が出てしまいます。AUCは**しきい値に依存せず、順位関係だけを見る**ので、確率のキャリブレーション（校正）がずれていても公平に比較できます。
- **AUCが順位指標であることの実務的な帰結**：最終提出でスコアを`rankdata`で[0,1]に変換しても**AUCは1ミリも変わりません**（順序が同じだから）。このnotebookの提出セルがまさにそれをやっています。逆に言えば、AUCを狙うなら確率の絶対値を合わせる努力は無駄で、**順位を正しくすることだけに集中すべき**です。
- **この手法の指標最適化設計**：
  - **rank-gauss変換**：各メンバーの予測を順位に直し、正規分布の分位点に写します。ライブラリによって出力レンジがバラバラ（あるものは[-14, +48]の生スコア）なので、順位に直して初めて比較可能になります。単なる順位だと分布の端の解像度が失われるため、ガウス分位点で端を引き伸ばしています。AUC基準で **+0.00008** の改善。
  - **nested（入れ子）CV**：5つの固定foldそれぞれについて、メタモデルを残り4foldで学習し、そのfoldで評価します。これをやらないと、メタモデルが自分の学習データを採点することになり、**OOFスコアが上振れします**。
  - **AUC上限の推定**：スコアをisotonic回帰でキャリブレーションしたうえで、`AUC* = E[1{p_i>p_j} p_i(1-p_j)] / (E[p]E[1-p])` を計算し、**約0.97006**という天井を出しています。実際のOOFが0.97001〜0.97009なので、伸びしろは5e-5程度。ラベルノイズ（真の確率が0.05〜0.95の行が36%、0.4〜0.6が5.8%）が原因で、そもそもコイン投げに近い行が大量にあります。


# S6E8: 177 models, and the ones that mattered were not the accurate ones

Public LB **0.97113**, from a stack over six public OOF libraries plus 22 models of my own.

**This runs end to end.** The pool is 155 members from the published OOF libraries **plus my
own 22**, which I have released as
[`s6e8-adarsh-oof-library`](https://www.kaggle.com/datasets/adarsh1077/s6e8-adarsh-oof-library)
(CC0) so the whole thing reproduces rather than stopping at the public half. Fork it and you
get the same number.

This is a writeup of what I measured, including the parts that failed. The headline:

> Once the public OOF pool is saturated, what buys you score is **disagreement, not accuracy**.
> Three of my four highest-weighted models are the ones I built by *removing* the target
> encoder. A plain logistic regression at AUC 0.9589 outranked eight stronger boosted trees.
> A seed twin of a model already in the pool took a **negative** coefficient.

**A correction, since this notebook is about measuring honestly.** Earlier versions of this
writeup claimed your own models are worth "~6x" a public member, from watching the LB move as
I added mine. That number does not survive a proper test. Section 7 runs leave-one-author-out
over the finished pool — drop everything one contributor published, refit, measure — and the
real figure is **~1.8x**, with one public library beating mine outright on value per array. The
direction holds; the magnitude was inflated by measuring at the moment of addition.

Everything below is on the frozen community fold split, so it stacks row-for-row with the
public libraries. Credit to @szymonkapiski, @boltuzamaki, @dariushafshar, @raykkretzschmar,
@beicicc, @mohankrishnathalla and @najiama, whose published OOF arrays are the backbone of the
pool, and to
@tomasa2, whose ablation notebook is where the feature recipe came from.

## 1. The fold convention, and why row order is load-bearing

Every public library on this competition aligns to one split:

```python
StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train, train.addicted_label)
# over train.csv in ORIGINAL FILE ROW ORDER - never sorted, never reindexed
```

There are no ids in most of the `.npy` files, so **alignment is positional**. If you sort or
filter the competition frame before aligning, every number you compute afterwards is wrong
and will still look plausible.

I did not take this on faith. Three libraries ship their own `fold_id.npy`; I checked all
three against the regenerated split and they are bit-identical (two are 1-indexed, which is
worth knowing before you compare them).

---
### セル1：foldを厳密に再現し、行順が壊れていないか検証する

**What**：`train.csv` / `test.csv` を読み、`StratifiedKFold(5, shuffle=True, random_state=42)` で分割を作り、他の参加者が公開している `fold_id.npy` と**一致するか照合**しています。

**Why**：このコンペでは大勢が「OOF予測の配列（.npy）」を公開し合っています。ところが `.npy` にはIDが入っていません。つまり**行の並び順だけが対応関係の唯一の手がかり**です。もし `train` を一度でもソートしたりフィルタしたりすれば、以降の全計算が静かに間違います。しかも**間違ったままそれらしい数字が出る**ので気づけません。

著者は「信じずに検証する」立場を取り、3つの公開`fold_id`をビット単位で照合しています（うち2つは1始まりのインデックスだった、という実務的な落とし穴も記録されています）。

> 補足：**OOF (Out-Of-Fold) 予測**とは、交差検証で「そのfoldを学習に使っていないモデル」が出した予測のこと。学習データ全行分の「カンニングなし予測」が得られるので、これを特徴量にして2段目のモデル（メタモデル）を学習するのが**スタッキング**です。


In [ ]:
import os, glob, hashlib
import numpy as np, pandas as pd
from scipy.stats import rankdata, norm, ks_2samp
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

COMP = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
D = os.path.dirname(COMP)
train = pd.read_csv(f"{D}/train.csv")          # do NOT sort or reindex
test = pd.read_csv(f"{D}/test.csv")
y = train["addicted_label"].to_numpy(np.int8)
NTR, NTE = len(train), len(test)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(NTR), y))
print(f"train {NTR:,}  test {NTE:,}  positive rate {y.mean():.4f}")

mine = np.empty(NTR, np.int8)
for i, (_, iva) in enumerate(FOLDS):
    mine[iva] = i
for p in glob.glob("/kaggle/input/**/*fold_id*.npy", recursive=True):
    f = np.load(p).astype(int)
    if f.min() == 1:
        f -= 1                      # some libraries ship 1-indexed fold labels
    print(f"  {os.path.basename(p):<40s} identical to frozen split: {bool((f == mine).all())}")

## 2. Building the pool

Six public libraries publish OOF + test pairs on this split. They use two naming conventions
(`oof_x.npy`/`test_x.npy` and `x_oof.npy`/`x_test.npy`) plus a couple of parquet tables, so
the loader handles all of them and aligns parquet rows by `id`.

---
### セル2：予測プールの読み込み（命名規則の吸収と出所の記録）

**What**：`/kaggle/input/` 以下を再帰的に走査し、`oof_x.npy` / `test_x.npy` 形式と `x_oof.npy` / `x_test.npy` 形式の両方、さらにparquetテーブルからも予測を集めます。形状が想定と違うものは黙って捨てます。同時に `PROV`（provenance＝出所）辞書に「このメンバーはどのデータセット由来か」を記録します。

**Why**：公開ライブラリは作者ごとに命名規則が違うので、片方だけ対応すると半分を取りこぼします。出所を記録しているのは、後半の「**leave-one-author-out**（ある作者の配列を全部抜いたらスコアがどれだけ落ちるか）」という分析のためです。メンバー名（`catnative` など）から作者は推測できないので、**ファイル名でなくデータセットのスラッグで帰属させる**という判断は真似する価値があります。

> 補足：`glob.glob(..., recursive=True)` の `**` はディレクトリを何段でもまたぐワイルドカード。Kaggleのように入力データの階層が不定な環境で便利です。


In [ ]:
PROV = {}   # member name -> the /kaggle/input dataset it came from, for §7's attribution

def load_pool():
    members, seen = {}, {}
    tr_id, te_id = train["id"].values, test["id"].values

    def add(name, o, t, src=""):
        o = np.asarray(o, np.float64).ravel(); t = np.asarray(t, np.float64).ravel()
        if o.shape != (NTR,) or t.shape != (NTE,):
            return
        members[name] = (o, t)
        # /kaggle/input/<dataset-slug>/... -> the slug identifies who published it
        parts = src.replace("\\", "/").split("/")
        PROV[name] = parts[3] if len(parts) > 3 else ""

    for p in glob.glob("/kaggle/input/**/*.npy", recursive=True):
        b = os.path.basename(p)[:-4]
        d = os.path.dirname(p)
        if b.startswith("oof_"):
            tp, key = os.path.join(d, f"test_{b[4:]}.npy"), b[4:]
        elif b.endswith("_oof"):
            tp, key = os.path.join(d, f"{b[:-4]}_test.npy"), b[:-4]
        else:
            continue
        if os.path.exists(tp):
            add(key, np.load(p), np.load(tp), p)

    for p in glob.glob("/kaggle/input/**/*oof*.parquet", recursive=True):
        # replace in the FILENAME only: the containing folder is often called
        # "s6e8-oof-prediction-library", and a whole-path replace mangles it into
        # "s6e8-test-prediction-library", which silently drops 47 members.
        tp = os.path.join(os.path.dirname(p), os.path.basename(p).replace("oof", "test"))
        if not os.path.exists(tp):
            continue
        do, dt = pd.read_parquet(p), pd.read_parquet(tp)
        if "id" not in do.columns:
            continue
        do = do.set_index("id").reindex(tr_id); dt = dt.set_index("id").reindex(te_id)
        for c in do.columns:
            if c in dt.columns:
                add(c, do[c].to_numpy(np.float64), dt[c].to_numpy(np.float64), p)

    # @najiama publishes blends as CSV pairs rather than .npy
    for p in glob.glob("/kaggle/input/**/*_blend_oof_predictions.csv", recursive=True):
        k = os.path.basename(p).split("_")[0]
        sp = os.path.join(os.path.dirname(p), f"{k}_blend_submission.csv")
        if not os.path.exists(sp):
            continue
        do, dt = pd.read_csv(p), pd.read_csv(sp)
        oc = [c for c in do.columns if c.lower() != "id"]
        tc = [c for c in dt.columns if c.lower() != "id"]
        if not oc or not tc:
            continue
        if "id" in do.columns:
            do = do.set_index("id").reindex(tr_id)
        if "id" in dt.columns:
            dt = dt.set_index("id").reindex(te_id)
        add(f"naji_blend{k}", do[oc[0]].to_numpy(np.float64),
            dt[tc[0]].to_numpy(np.float64), p)
    return members

pool = load_pool()
print(f"collected {len(pool)} members")

## 3. Quarantine — and two arrays that are published twice

Three checks, in order of how much trouble they save:

1. **Exact duplicates.** Hash every OOF array. A duplicate silently *doubles* that model's
   weight in the meta-model. I found two: one array appears in two different beicicc
   datasets under different names, and `xgb_d7_alt1`/`xgb_d7_alt2` are byte-identical
   **inside the same library**. Neither is anybody's mistake to be embarrassed about — but
   if you glob a directory and stack whatever you find, you will double-count them.
2. **OOF↔test distribution drift**, measured as a KS statistic on the rank scale. This is
   @dariushafshar's idea. Three members fail it here: `knn` (0.180), `rf` (0.081) and
   `extratrees_support` (0.073).
3. **Degenerate members** (non-finite, or AUC below 0.90) — *except* deliberate
   correctors. `sixmember_meta_perp` is a frozen geometry-residual **direction**, not a
   prediction; its solo AUC is 0.469 and it is meant to be used with a signed coefficient.
   A naive "drop anything under 0.9" rule throws it away.

---
### セル3：検疫（quarantine）— 重複・分布ドリフト・劣化メンバーの除去

**What**：3つのチェックを順に掛けます。

1. **完全重複の除去**：全OOF配列をMD5でハッシュ化し、同じものを削ります。実際に2組見つかっています（別々のデータセットに同じ配列が入っていた例と、同一ライブラリ内で `xgb_d7_alt1` と `xgb_d7_alt2` がバイト単位で同一だった例）。
2. **OOF↔test の分布ドリフト**：予測を順位スケールに直し、`ks_2samp`（コルモゴロフ–スミルノフ検定）でOOFとtestの分布の乖離を測ります。0.05を超えたら除外（`knn` 0.180、`rf` 0.081、`extratrees_support` 0.073 が該当）。
3. **劣化メンバー**：非有限値を含む、またはAUCが0.90未満のものを除外。ただし名前に `perp` を含む「補正役」は例外扱い。

**Why**：
- **重複を残すと、そのモデルの重みが黙って2倍になります。** メタモデルは「同じ意見が2票入った」と解釈してしまう。
- **ドリフトチェックが必要な理由**：OOFでは良いスコアなのに、testでは予測の分布が違う — というメンバーは、testで思ったとおりに振る舞いません。メタモデルの重みはOOFで決まるので、testで挙動が変わるメンバーに大きな重みが乗ると、LBだけが落ちます。
- **補正役を例外にする理由**：`sixmember_meta_perp` のようなメンバーは、単体では予測性能が無くても「他のメンバーが揃って外す方向」を表す幾何的な残差ベクトルです。AUCで足切りすると捨ててしまいます。**「単体性能が低い＝役立たず」ではない**というのが、このnotebook全体の主張そのものです。

> 補足：**KS統計量**は2つの分布の累積分布関数の最大差。0に近いほど同じ分布。分布シフトの検出によく使われます。


In [ ]:
seen, dups = {}, []
for n in sorted(pool):
    h = hashlib.md5(np.ascontiguousarray(pool[n][0]).tobytes()).hexdigest()
    if h in seen:
        dups.append((n, seen[h])); del pool[n]
    else:
        seen[h] = n
print(f"exact duplicates removed: {len(dups)}")
for a, b in dups:
    print(f"   {a}  ==  {b}")

R = lambda v: (rankdata(v, method="average") - 0.5) / len(v)
rng = np.random.default_rng(0)
ia, ib = rng.choice(NTR, 40000, False), rng.choice(NTE, 40000, False)
keep, dropped = [], []
for n, (o, t) in pool.items():
    au = roc_auc_score(y, o)
    corrector = "perp" in n
    if not (np.isfinite(o).all() and np.isfinite(t).all()):
        dropped.append((n, "nonfinite", au)); continue
    if au < 0.90 and not corrector:
        dropped.append((n, "auc<0.90", au)); continue
    ks = ks_2samp(R(o)[ia], R(t)[ib]).statistic
    if ks > 0.05 and not corrector:
        dropped.append((n, f"ks={ks:.3f}", au)); continue
    keep.append(n)
print(f"\nkept {len(keep)}, dropped {len(dropped)}")
for n, w, au in dropped:
    print(f"   drop {n:<28s} {w:<12s} auc={au:.6f}")

## 4. The meta-model

Rank-transform each member, push through the normal quantile (`rank-gauss`), then an L2
logistic regression, fitted **nested**: for each of the five frozen folds the meta-model is
fitted on the other four and scored on the held-out one, so no weight is ever fitted on a row
it is scored on.

Three details that are not cosmetic:

* **Rank-gauss beat logit space by +0.00008** on this pool. Members arrive on wildly
  different calibrations (one library ships raw FM scores in `[-14, +48]`); ranking makes
  them commensurable, and the gaussian tail restores the resolution that plain ranks lose
  where the target saturates.
* **`StandardScaler` is mandatory.** Without it lbfgs does not converge at any sane
  `max_iter`, and — this is the trap — **a non-converged fit reads HIGHER than the truth**.
  Assert `max(n_iter_) < max_iter` rather than hoping.
* **Regularisation barely matters.** `C` from 0.03 to 0.3 moves the nested score by 4e-6.
  The top is flat; picking its exact argmax is fitting noise.

One honesty note on the pool itself. A few published members (@najiama's `*_blend_*` files)
are **themselves blends fitted over this same pool**. They are excellent — the strongest
single members here — but their OOF is optimistic in-sample, so any cross-validation number
that includes them reads better than it generalises. The cell below prints the score **both
ways**: with them (what actually scores on the LB) and without them (what I would quote as
the honest number). The gap is small, and knowing its size is the point.

---
### セル4：メタモデル（rank-gauss → 標準化 → L2ロジスティック回帰、nested評価）

**What**：各メンバーを順位→正規分位点に変換し、`StandardScaler` で標準化してから `LogisticRegression(C=0.03)` で束ねます。評価は入れ子CV（5つの固定foldで、学習4fold・評価1fold）。最後に **収束したかを `assert` で検証**しています。

**Why（3つの実務的な勘所）**：

1. **rank-gauss**：メンバーごとに出力スケールが違うので、順位に直して初めて足し合わせられます。ただ順位のままだと分布の端（極端に高い／低いスコア）の情報が潰れるので、正規分位点に写して端の解像度を戻します。
2. **StandardScaler は必須**：これを外すと、lbfgsソルバーが現実的な `max_iter` で収束しません。しかも**収束しなかった場合、スコアは真の値より「高く」出ます**（正則化が効き切っていない＝過学習気味の重みで採点される）。だから「たぶん大丈夫」ではなく `assert max(n_iter_) < max_iter` で機械的に検証しています。
3. **正則化 `C` はほぼ効かない**：0.003〜1.0でスイープしても変化は ±0.000004。177本もあると個々の重みの微調整は誤差に埋もれます。**チューニングすべき場所とそうでない場所を、測ってから判断している**のがこのnotebookの姿勢です。

**この時点の比較**：単体最強メンバー < 等重み順位平均 < スタック、という並びが出力されます。等重み平均はスタックに **−0.0027** 負けます。

> 補足：`C` はロジスティック回帰の正則化の**逆数**の強さ。小さいほど強く正則化されます（scikit-learnの慣習）。


In [ ]:
gauss = lambda v: norm.ppf(np.clip(R(v), 1e-7, 1 - 1e-7))
G = np.column_stack([gauss(pool[n][0]) for n in keep])
Gt = np.column_stack([gauss(pool[n][1]) for n in keep])

def nested(X, C=0.03):
    pred = np.zeros(NTR); nit = []
    for itr, iva in FOLDS:
        sc = StandardScaler().fit(X[itr])
        m = LogisticRegression(C=C, max_iter=5000, solver="lbfgs", tol=1e-5)
        m.fit(sc.transform(X[itr]), y[itr])
        nit.append(int(np.max(m.n_iter_)))
        pred[iva] = m.decision_function(sc.transform(X[iva]))
    assert max(nit) < 5000, "not converged - this score would read HIGH"
    return roc_auc_score(y, pred), pred

best_single = max(roc_auc_score(y, pool[n][0]) for n in keep)
auc_mean = roc_auc_score(y, np.column_stack([R(pool[n][0]) for n in keep]).mean(1))
auc_stack, _ = nested(G)
print(f"best single member          {best_single:.6f}")
print(f"equal rank average          {auc_mean:.6f}")
print(f"nested rank-gauss stack     {auc_stack:.6f}   <- FULL pool")

# Some published members are themselves BLENDS fitted over this same pool (naji's). Their OOF
# is optimistic in-sample, so the honest comparison excludes them. Both are reported: the
# clean number is the one to trust, the full number is the one that scores.
base = [n for n in keep if not n.startswith("naji_blend")]
if len(base) < len(keep):
    Gb = np.column_stack([gauss(pool[n][0]) for n in base])
    auc_base, _ = nested(Gb)
    print(f"nested, pre-blends EXCLUDED {auc_base:.6f}   <- honest ({len(base)} members)")
    print(f"difference                  {auc_stack - auc_base:+.6f}")

## 5. The result that matters: diversity beats strength

I trained 22 models of my own on the same folds (imputation kept *alongside* the raw NaN
columns, composition ratios, the decimal lattice, and 10-fold nested target+frequency
encoding of every column — the recipe from @tomasa2's ablation).

Adding five of them took the nested score 0.970043 → 0.970081 and the **leaderboard 0.97106 →
0.97113**, moving me from rank 20 to rank 6. Then I kept going to 22 models, which is where the
pattern becomes unmistakable. Ranked by the coefficient the meta-model assigned, *not* by
accuracy:

| my model | solo OOF | coefficient |
|---|---|---|
| CatBoost, its own ordered target statistics | 0.968601 | **+0.749** |
| XGBoost, **no target encoding** | 0.967042 | **+0.334** |
| LightGBM, **no target encoding** | 0.966381 | **+0.278** |
| CatBoost, **no target encoding** | 0.968405 | **+0.176** |
| XGBoost + TE | 0.968372 | +0.158 |
| **logistic regression** (yes, really) | **0.958910** | **+0.137** |
| … | | |
| LightGBM + TE, **different seed** | 0.968155 | **+0.004** |
| CatBoost, seed 7 | 0.968093 | −0.095 |
| LightGBM, colsample 0.3 | 0.968301 | −0.127 |
| HistGradientBoosting | 0.967468 | −0.129 |

**Three of the top four are the no-encoder views** — arrived at independently by XGBoost,
LightGBM and CatBoost. A plain logistic regression, the second-weakest model I own, outranked
eight stronger boosted trees. And the bottom of the table is entirely *seeds and
hyperparameter tweaks of recipes already represented* — the stack uses those as small
negative corrections, not as things to average in.

The rule that falls out: **train different views (drop the encoder, change the family), never
more seeds.** Measured: four extra XGBoost variants were worth +0.000003 between them, and
four extra CatBoost variants after the first were worth **−0.000001**.

All 22 arrays are in the linked dataset, weak and redundant ones included, so this is
reproducible rather than a claim.

## 6. What did not work — measured, same folds

| idea | nested delta |
|---|---|
| growing the pool 79 → 132 members | **+0.00032** |
| rank-gauss instead of logit meta-features | +0.00008 |
| `C` swept over 0.003 – 1.0 | ±0.000004 (flat) |
| riponce/szymon **missingness-regime interactions** | **−0.000008** (fell back to global) |
| greedy hill-climb instead of logistic regression | −0.00019 |
| equal rank average | −0.0027 |

The regime design is the interesting failure: it is a good idea, it is measured to help on
other people's pools, and on mine it did nothing. Per-bucket it *helped* the complete-rows
bucket (+0.00002) and hurt the other two. Re-measure inherited tricks; do not adopt them.

**And once your stack outgrows the public files, stop blending with them.** At LB 0.97106,
blending my stack with the strong public submissions helped. At 0.97113 the pure stack
**beat** the same blend (0.97113 vs 0.97111). Re-test the blend every time the stack improves.

## 7. Two avenues I closed, so you don't have to

**There is no exact-match / lookup channel.** All **691,369 train rows have distinct
12-column keys** (NaN as its own level) and **0.00 % of test rows match any train row**.
Dropping any single column still leaves at most 30 rows in a repeated key. The strong public
`lookup_*` members are therefore not doing row retrieval — they are per-column exact-value
target encoding, which is the same channel `te_<col>` already covers.

**No further generator-fingerprint feature survives.** The first decimal digit genuinely works
(it is in the pool). I tested 35 more candidates on top of the stack's own OOF logit — the
generator's slack term `other = daily − social − gaming − work` with its fraction and digits,
the **second** decimal, both decimals as one level, integer divisibility mod 2/5/10, and
cross-column digit interactions. Together: **−0.000182**. Noise, and the wrong sign.

The shape of that experiment is the reusable part: **regress candidates against what the
current stack still gets wrong**, not against the raw target. A feature can look informative
marginally and carry nothing the ensemble has not already extracted.

---
### セル5：「完全一致ルックアップ」チャネルが存在しないことの証明

**What**：全12特徴量の値を1つのハッシュキーに畳み込み（NaNも独立のレベルとして扱う）、train内でキーが何通りあるか、testの行がtrainのキーと何%一致するかを数えています。

**Why**：合成データのコンペでは、「trainとtestに同じ行が混じっていて、単純に引き当てるだけで高得点」というショートカットが存在することがあります。著者はこれを潰しました：**691,369行すべてが異なるキーを持ち、testの一致率は0.00%**。どの1列を落としても、同じキーを共有する行は最大30行しか出ません。

つまり、公開されている `lookup_*` という名前のメンバーは**行の検索をしているのではなく、列ごとの値に対するtarget encodingをしているだけ**だと結論づけています。

**学べる姿勢**：「他人のメンバー名から手法を推測する」のではなく、**自分でデータを叩いて確かめる**。10行のコードで、週末をひとつ節約できます。

> 補足：`pd.factorize(..., use_na_sentinel=False)` は、NaNを「欠損」ではなく**ひとつのカテゴリ**として符号化します。NaNパターン自体が情報を持つデータではこれが正解です。


In [ ]:
F = [c for c in test.columns if c != "id"]
def keyhash(df_cols):
    h = np.zeros(NTR + NTE, np.uint64)
    both = pd.concat([train[F], test[F]], ignore_index=True)
    for c in df_cols:
        cod, _ = pd.factorize(both[c], use_na_sentinel=False)
        h = h * np.uint64(1000003) + cod.astype(np.uint64) + np.uint64(0x9E3779B9)
    return h
h = keyhash(F)
htr, hte = h[:NTR], h[NTR:]
print(f"distinct train keys : {len(np.unique(htr)):,} of {NTR:,}")
print(f"test rows matching a train row: {pd.Index(hte).isin(pd.Index(np.unique(htr))).mean():.2%}")

## 8. How high can *anyone* score here? Roughly 0.9701 out-of-fold.

Worth knowing before you spend another weekend on this. The target is a Bernoulli draw from a
smooth probability field, so there is a hard AUC ceiling, and a **calibrated** model lets you
estimate it:

$$\text{AUC}^{*} = \frac{\mathbb{E}\left[\mathbb{1}\{p_i > p_j\}\, p_i (1-p_j)\right]}{\mathbb{E}[p]\,\mathbb{E}[1-p]}$$

Calibrate the stack out-of-fold with isotonic regression, check the reliability table, then
evaluate that by sorting rather than pairwise. On my stack it comes out at **0.97006**, versus
a stack already at 0.97001-0.97009 — **headroom of about 5e-5, and possibly none**.

The cap is irreducible label noise: **36% of rows have a true probability between 0.05 and
0.95**, and 5.8% sit between 0.4 and 0.6, where the label is genuinely a coin flip given the
features. So if you were wondering whether 0.98 is out there — it is roughly **180x the
entire remaining headroom**.

One honest caveat: this bounds signal reachable *from the current representation*. @tomasa2
watched it under-predict by ~1e-4 when the decimal-lattice channel turned up. Use it to stop
tuning, never to stop looking.

---
### セル6：AUCの理論上限（Bayes AUC）の推定

**What**：
1. スタックの出力を **isotonic回帰**でout-of-foldにキャリブレーションし、真の確率 `p` の推定値を得ます。
2. 十分位ごとの「予測平均 vs 実測平均」の表（reliability table）を出して、キャリブレーションが妥当かを目視確認。
3. `bayes_auc()` で、その確率場から得られる**AUCの理論最大値**を計算します。

**Why**：ターゲットが「滑らかな確率場からのベルヌーイ試行」である場合、どんな完璧なモデルでも到達できない天井があります。真の確率が 0.45 の行と 0.55 の行を完璧に順位づけても、実際のラベルはコイン投げなので必ず一定割合で逆転するからです。

計算結果は **AUC\* ≈ 0.97006**。現在のスタックが 0.97001〜0.97009 なので、**残りの伸びしろは 5e-5 程度、あるいはゼロ**。

**なぜこの計算が実用的か**：「あと2週間かけて0.0002伸ばす」という計画が、**そもそも物理的に不可能かどうか**を10分で判定できます。Kaggleに限らず、機械学習プロジェクトで「これ以上の精度向上は投資に見合うか」を答えるための強力な道具です。

**式の意味**：

$$\text{AUC}^{*} = \frac{\mathbb{E}[\mathbb{1}\{p_i > p_j\}\, p_i (1-p_j)]}{\mathbb{E}[p]\,\mathbb{E}[1-p]}$$

分子は「真の確率が高い方の行が実際に陽性で、低い方が実際に陰性になる確率」。分母は正規化項。実装では全ペア（O(n²)）を回さず、`p` をソートして累積和で O(n log n) に落としています。

> 補足：**isotonic回帰**は「単調増加であること」だけを仮定するノンパラメトリックな校正手法。ロジスティック回帰による校正（Platt scaling）より柔軟ですが、データが少ないと過学習します。ここは69万行あるので問題ありません。


In [ ]:
from sklearn.isotonic import IsotonicRegression
_, raw = nested(G)                      # nested stack scores, no row scored by its own fit
p = np.zeros(NTR)
for itr, iva in FOLDS:
    iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6)
    p[iva] = iso.fit(raw[itr], y[itr]).predict(raw[iva])
p = np.clip(p, 1e-6, 1 - 1e-6)

tab = pd.DataFrame({"b": pd.qcut(p, 10, labels=False, duplicates="drop"), "p": p, "y": y}) \
        .groupby("b").agg(n=("y", "size"), predicted=("p", "mean"), observed=("y", "mean"))
print(tab.to_string(float_format="%.4f"))
print(f"max |predicted - observed| = {(tab.predicted - tab.observed).abs().max():.4f}\n")

def bayes_auc(p):
    p = np.sort(np.asarray(p, np.float64)); n = len(p); comp = 1.0 - p
    cum = np.concatenate([[0.0], np.cumsum(comp)[:-1]])
    return (float(np.dot(p, cum)) / (n * n)) / float(p.mean() * comp.mean())

ceil = bayes_auc(p)
print(f"estimated Bayes-optimal AUC = {ceil:.6f}")
print(f"this stack (out-of-fold)    = {roc_auc_score(y, raw):.6f}")
print(f"headroom                    = {ceil - roc_auc_score(y, raw):+.6f}")
for lo, hi in [(0.05, 0.95), (0.4, 0.6)]:
    print(f"rows with true p in ({lo},{hi}): {((p > lo) & (p < hi)).mean():.1%}")

## 9. CV → LB

Trust CV for **ranking** candidates, never as a leaderboard estimate. Across six submissions
the leaderboard sat a stable **+0.00098 to +0.00115** above the nested OOF, because every test
prediction is an average of five fold-models while every OOF row comes from one. The offset is
harmless. A wrong *ordering* would not be.

## 10. Submission

---
### セル7：最終提出（全データでメタモデルを再学習 → 順位に変換）

**What**：全trainでメタモデルを1回学習し、testに適用。その出力を `rankdata` で[0,1]の順位に変換して `submission.csv` に書き出します。最後に、最も負の係数（＝補正役）と最も正の係数のメンバーを表示します。

**Why（順位変換をする理由）**：AUCは順序しか見ないので、順位に変換しても**スコアは変わりません**。それでもやるのは、(a) 出力範囲が[0,1]に収まって提出フォーマットの検証が通りやすい、(b) 極端な値によるオーバーフローや数値の見づらさを避けられる、という実務上の理由からです。

**Why（`assert` を挟む理由）**：行数が一致しているか、全て有限値か、収束したか — を提出前に機械的に確認しています。Kaggleは1日の提出回数に上限があるので、フォーマットミスで1回無駄にするのは高くつきます。

**係数表示の意味**：**最も負の係数を持つメンバー = 補正役**です。「このメンバーが高いと言っているときは、むしろ下げる」という使われ方をしている。単体で見れば予測性能の低いモデルが、こういう役割で貢献します。


In [ ]:
sc = StandardScaler().fit(G)
meta = LogisticRegression(C=0.03, max_iter=5000, solver="lbfgs", tol=1e-5).fit(sc.transform(G), y)
assert int(np.max(meta.n_iter_)) < 5000
pred = R(meta.decision_function(sc.transform(Gt)))          # rank -> [0,1]; AUC only sees order

sub = pd.DataFrame({"id": test["id"].values, "addicted_label": pred})
assert len(sub) == NTE and np.isfinite(sub["addicted_label"]).all()
sub.to_csv("submission.csv", index=False)
print(sub.head())
print(f"\nrows {len(sub):,}  distinct {sub['addicted_label'].nunique():,}")

coef = pd.Series(meta.coef_[0], index=keep).sort_values()
print("\nmost negative coefficients (the correctors):")
print(coef.head(5).round(3).to_string())
print("\nlargest positive:")
print(coef.tail(8).round(3).to_string())

## 7. What is each OOF library actually worth? Leave-one-author-out.

Everyone in this competition is stacking the same shared pool, and nobody has measured what
any part of it is worth. So: for each contributor, **drop every array they published, refit
the stack, and record what is lost.** Same folds, same C, 178 members.

| contributor | arrays | nested loss when dropped | per array |
|---|---|---|---|
| @boltuzamaki | 45 | **+0.000189** | 4.2e-6 |
| mine | 22 | **+0.000057** | 2.6e-6 |
| @szymonkapiski | 67 | +0.000016 | 0.2e-6 |
| @najiama | 14 | +0.000010 | 0.7e-6 |
| @mohankrishnathalla | 4 | +0.000004 | 1.0e-6 |
| @raykkretzschmar | 5 | +0.000002 | 0.4e-6 |
| @beicicc | 14 | +0.000002 | 0.1e-6 |
| @dariushafshar | 7 | −0.000001 | −0.1e-6 |

**Read the bottom six rows as a single "indistinguishable from zero" band, not as a ranking.**
Differences of ~1e-6 on a 691k-row nested AUC are below what this setup resolves, and it would
be wrong to order people by them. Only the top two rows are clearly above noise.

Two things worth taking away:

**Library size does not predict library value.** The 67-array library is the largest in the
pool and sits in the noise band; the 45-array one is worth *twelve times* more. A big library
of mutually-correlated members is reconstructible from the rest of the pool, so dropping it
costs almost nothing. This is the same finding as §5 — diversity over strength — one level up:
it applies to whole libraries, not just to individual models.

**A leave-one-out is harsh on blends.** Several contributors published *pre-blends* — ensembles
of models that remain in the pool after the blend is dropped, so the stack simply rebuilds most
of what they were doing. Their figures here are a floor, not a valuation. Base models carry no
such handicap. Comparing the two kinds directly would not be fair, and I am not doing it.

And the correction from the top of this notebook, stated plainly: dropping all 22 of my own
models costs 0.000057, or 2.6e-6 per array, against a public-pool average of 1.4e-6. That is
**1.8x, not the ~6x I claimed earlier** — and @boltuzamaki's arrays beat mine at 4.2e-6. My
own models were not the most valuable thing in this pool. Someone else's were.

---
### セル8：leave-one-author-out アブレーション（既定ではOFF）

**What**：各投稿者について「その人が公開した配列を全部抜いてスタックを組み直し、スコアがどれだけ落ちるか」を測ります。9回フルに学習し直すので約14分かかるため、`RUN_ABLATION = False` で既定オフになっています。

**Why**：全員が同じ共有プールをスタックしているのに、**その中身が実際いくらの価値を持つのかを誰も測っていなかった**、というのが著者の問題意識です。結果：

| 投稿者 | 配列数 | 抜いたときの損失 | 1配列あたり |
|---|---|---|---|
| @boltuzamaki | 45 | +0.000189 | 4.2e-6 |
| 著者自身 | 22 | +0.000057 | 2.6e-6 |
| @szymonkapiski | 67 | +0.000016 | 0.2e-6 |
| （以下6名） | — | ~1e-6以下 | — |

著者は**下位6行を「ゼロと区別がつかない帯」であって順位ではない**と明記しています。69万行のnested AUCで1e-6の差は、fold分割の揺らぎに埋もれるからです。これは統計的な誠実さの手本で、Kaggleの公開notebookでは珍しい態度です。

**注意点として書かれていること**：帰属は**メンバー名でなく、読み込み元のデータセットスラッグ**で行うべき。配列名は `catnative` のように素っ気なく、あるライブラリは投稿者ではなくモデル名で命名されているため、名前で判断すると別人にクレジットしてしまいます。

> 補足：**アブレーション（ablation study）**は「構成要素を1つずつ取り除いて、性能への寄与を測る」実験手法。論文でもよく使われます。「入れたら上がった」ではなく「抜いたら下がった」を測るほうが、寄与の証明として強くなります。


In [ ]:
# The ablation above. Off by default: it refits the full stack nine times (~14 min).
RUN_ABLATION = False

if RUN_ABLATION:
    # Attribute by the dataset a member was loaded from, NOT by its name: the arrays carry
    # bare keys like "catnative", and one library ("golem") is named after the model rather
    # than its publisher, which is an easy way to credit the wrong person.
    OWNER = {
        "s6e8-oof-library-47-models":                       "szymonkapiski",
        "s6e8-oof-prediction-library":                      "boltuzamaki",
        "s6e8-golem-oof-library":                           "dariushafshar",
        "s6e8-measured-findings-pack":                      "dariushafshar",
        "s6e8-fm-lattice-blend-members":                    "raykkretzschmar",
        "predicting-smartphone-addiction-oof-submission-csv": "najiama",
        "s6e8-adarsh-oof-library":                          "mine",
    }

    def author(n):
        src = PROV.get(n, "")
        if src in OWNER:
            return OWNER[src]
        if src.startswith("s6e8-") and src.endswith("-oof"):
            return "mohankrishnathalla"
        if src.startswith("s6e8-cat-mlp") or src.startswith("s6e8-lgb-dart"):
            return "mohankrishnathalla"
        if src.startswith("s6e8-"):
            return "beicicc"      # every remaining attached library is beicicc's
        return "unattributed"

    by_auth = {}
    for n in keep:
        by_auth.setdefault(author(n), []).append(n)
    # naji's arrays ship inside szymon's library too; count them to naji
    for n in keep:
        if n.startswith("naji"):
            for a in by_auth:
                if n in by_auth[a] and a != "najiama":
                    by_auth[a].remove(n); by_auth.setdefault("najiama", []).append(n)
    assert not by_auth.get("unattributed"), by_auth.get("unattributed")

    full_auc, _ = nested(G)
    print(f"full pool ({len(keep)}): {full_auc:.6f}")
    for a in sorted(by_auth, key=lambda k: -len(by_auth[k])):
        cols = [i for i, n in enumerate(keep) if author(n) != a]
        au, _ = nested(G[:, cols])
        print(f"  without {a:<16s} ({len(by_auth[a]):>3d}) {au:.6f}  loss={full_auc-au:+.6f}")

## Takeaways

1. **Hunt for more members before you tune the combiner.** Pool growth was worth +0.00032;
   every meta-model refinement I tried was worth zero or less.
2. **Then stop, and train your own — for disagreement, not accuracy.** Public members
   correlate 0.99+ with each other, so an independent model is worth ~1.8x one more of
   theirs (leave-one-author-out, §7 — and note that one public library beat mine on that
   measure). Different *views* beat different seeds, decisively.
3. **Hash your arrays before stacking them.** Two published arrays here are duplicates.
4. **Assert your meta-model converged.** A non-converged logistic regression reads higher
   than the truth, silently.
5. **Test new features against the stack's residuals**, not against the target.
6. **Estimate the ceiling before you grind.** Ten minutes of arithmetic tells you whether the
   0.0002 you are chasing exists at all. Here it mostly does not — the top of this
   leaderboard is separated by less than the metric can cleanly resolve, so expect the
   private split to reshuffle it.

Corrections welcome — if something here does not reproduce for you, I would rather know.